In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
# Question 1:

In [3]:
first_3_pages = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]
documents_llm = []
for doc in documents:
    if doc["filename"] in first_3_pages:
        documents_llm.append(doc)
len(documents_llm)

3

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
import os

openai_client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

In [7]:
import json
input_tokens_list = list()
for doc in documents_llm:
    user_prompt = json.dumps(doc)
    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.beta.chat.completions.parse(
        model="openai/gpt-5",
        messages=messages,
        response_format=Questions
    )
    input_tokens_list.append(response.usage.prompt_tokens)

In [8]:
sum(input_tokens_list) / len(input_tokens_list)

1356.0

In [9]:
# the full ground truth and searching the chunks

In [10]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth.to_dict(orient='records')
ground_truth[:10]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/lessons/02-environmen

In [11]:
from embedder import Embedder
model = Embedder()

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

In [14]:
texts = []

for doc in chunks:
    texts.append(doc["content"])

vectors = model.encode_batch(texts)

In [15]:
import numpy as np

X = np.array(vectors)

In [16]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

In [17]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    vector = model.encode(query)
    return vindex.search(vector, num_results=num_results)

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [20]:
# Question 2

In [21]:
q = ground_truth[0]["question"]

In [22]:
q = ground_truth[0]["question"]
text_search(q)[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [23]:
# Question 3

In [24]:
vector_search(q)[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [25]:
def compute_relevance(q, search_function):
    file_name = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == file_name))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in ground_truth:
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [26]:
# Question 4

In [27]:
evaluate(ground_truth, text_search)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [28]:
# Question 5

In [29]:
evaluate(ground_truth, vector_search)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [30]:
# Question 6

In [31]:
results = []
k_list = [1, 50, 100, 200]
for k in k_list:
    result = evaluate(ground_truth, hybrid_search)
    results.append({
        "k": k,
        "mrr": result["mrr"],
        "hit_rate": result["hit_rate"]
    })

results = pd.DataFrame(results)
results.sort_values(by=["mrr", "k"], ascending=[False, True]).head(1)

,k,mrr,hit_rate
0,1,0.637917,0.836111
